# Workshop 4

We will be building and training a basic character-level Recurrent Neural
Network (RNN) to classify words. This lab is based on Based on ["NLP From Scratch: Classifying Names with a Character-Level RNN"](https://pytorch.org/tutorials/intermediate/char_rnn_classification_tutorial.html) by [Sean Robertson](https://github.com/spro).

A character-level RNN reads words as a series of characters and for each one it (a) produces an output, and (b) updates a hidden state vector. The output hidden state from one step is in the input to the next step. In this lab, the final prediction will be made based on the last output.

The task we'll consider is predicting the language of origin of a name.
We'll train on a few thousand surnames from 18 languages
of origin, and predict which language a name is from based on the
spelling. For example:

```sh
$ python predict.py Hinton
(-0.47) Scottish
(-1.52) English
(-3.57) Irish

$ python predict.py Schmidhuber
(-0.19) German
(-2.48) Czech
(-2.68) Dutch
```

# Pre-Work

## Preparing the Data

Included in the ``names.txt`` file are a list of names, one per line, and a language they are commonly used in.
We have converted them to ASCII for convenience.

We'll read them in and make a dictionary of lists of names per language, ``{language: [names ...]}``.

In [14]:
import string
all_letters = string.ascii_letters + " .,;’"
n_letters = len(all_letters)

# Read the category_lines dictionary, a list of names per language
category_lines = {}
all_categories = set()

with open("names.txt") as src:
    for line in src:
        parts = line.strip().split()
        category = parts[0]
        name = ' '.join(parts[1:])
        all_categories.add(category)
        category_lines.setdefault(category, []).append(name)
    
all_categories = sorted(list(all_categories))
n_categories = len(all_categories)

Now we have ``category_lines``, a dictionary mapping each category
(language) to a list of lines (names). We also kept track of
``all_categories`` (just a list of languages) and ``n_categories`` for
later reference.




In [15]:
print(category_lines['Italian'][:5])

### Turning Names into Tensors

Now that we have all the names organized, we need to turn them into
Tensors to make any use of them.

To represent a single letter, we use a one-hot vector of size
`<1 x n_letters>`. A one-hot vector is filled with 0s except for a 1
at index of the current letter, e.g. `"b" = <0 1 0 0 0 ...>`.
In lecture 1, we noted that usually we use special data structures to avoid memory overhead (e.g., dictionaries or sparse vectors).
In this lab, we'll use a normal vector for convenience.

To make a word we join a bunch of those into a 2D matrix
``<line_length x 1 x n_letters>``.

That extra 1 dimension is because PyTorch assumes everything is in
batches - we're just using a batch size of 1 here.




In [16]:
import torch
torch.random.manual_seed(0)

# Find letter index from all_letters, e.g. "a" = 0
def letterToIndex(letter):
    return all_letters.find(letter)

# Just for demonstration, turn a letter into a <1 x n_letters> Tensor
def letterToTensor(letter):
    tensor = torch.zeros(1, n_letters)
    tensor[0][letterToIndex(letter)] = 1
    return tensor

# Turn a line into a <line_length x 1 x n_letters>,
# or an array of one-hot letter vectors
def lineToTensor(line):
    tensor = torch.zeros(len(line), 1, n_letters)
    for li, letter in enumerate(line):
        tensor[li][0][letterToIndex(letter)] = 1
    return tensor

print("This is the tensor for 'J':")
print(letterToTensor('J'))

print("\nThis is the dimensionality of the matrix for 'Jones':")
print(lineToTensor('Jones').size())

## Creating the Network

Before autograd, creating a recurrent neural network in Torch involved
cloning the parameters of a layer over several timesteps. The layers
held hidden state and gradients which are now entirely handled by the
graph itself. This means you can implement a RNN in a very "pure" way,
as regular feed-forward layers.

This RNN module is just 2 linear layers which operate on an input and hidden state, with
a ``LogSoftmax`` layer after the output.




In [17]:
import torch.nn as nn

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RNN, self).__init__()

        self.hidden_size = hidden_size

        # Define the layers of the model
        # These also create the weights where needed
        self.i2h = nn.Linear(input_size + hidden_size, hidden_size)
        self.h2o = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

        # Set the weights to some initial values
        self.init_weights()
        
    def init_weights(self):
        # Initialise the weights to be random values in the matrices and zero for the biases
        initrange = 0.1
        self.i2h.weight.data.uniform_(-initrange, initrange)
        self.i2h.bias.data.zero_()
        self.h2o.weight.data.uniform_(-initrange, initrange)
        self.h2o.bias.data.zero_()
        
    def initHidden(self):
        # Define the initial hidden state
        # Here we use an all zero vector
        return torch.zeros(1, self.hidden_size)

    def forward(self, input_tensor, hidden):
        # Given an input, compute the steps defined by the model

        # Concatenate the input and hidden vectors
        combined = torch.cat((input_tensor, hidden), 1)
        
        # Apply a linear layer to get the new hidden vector
        hidden = self.i2h(combined)

        # Apply a linear layer to get the output scores
        output = self.h2o(hidden)

        # Use softmax to turn the scores into probabilities
        output = self.softmax(output)
        return output, hidden

n_hidden = 128
rnn = RNN(n_letters, n_hidden, n_categories)

To run a step of this network we need to pass an input (in our case, the
Tensor for the current letter) and a previous hidden state (which we
initialize as zeros at first). We'll get back the output (probability of
each language) and a next hidden state (which we keep for the next
step).

Note - we haven't trained the model yet, so it's outputs will be random.



In [18]:
input_tensor = letterToTensor('A')
hidden = torch.zeros(1, n_hidden)

output, next_hidden = rnn(input_tensor, hidden)

For the sake of efficiency we don't want to be creating a new Tensor for
every step, so we will use ``lineToTensor`` instead of
``letterToTensor`` and use slices. This could be further optimized by
precomputing batches of Tensors.




In [19]:
print("Running 'Albert' through the RNN, Step-by-Step")
line_tensor = lineToTensor('Albert')
hidden = torch.zeros(1, n_hidden)

# Pass character "A" (line_tensor[0]) into the rnn to get
# - the "output" (nLikelihood for each label for given character 'A')
# - the "next_hidden" state (hidden state for processing the next character 'l')
output, next_hidden = rnn(line_tensor[0], hidden)

print("\nFor given characters 'A'")
print(f"\nNext hidden state <{next_hidden.size()}> is:")
print(next_hidden)

print(f"\nLikelihood <{output.size()}> for each label is:")
print(output)
print("-"*20)
# Pass character "l" (line_tensor[1]), and next_hidden into the rnn to get
# - the output (nLikelihood for each label for given character 'A' and 'l')
# - the next hidden state (hidden state for processing the next character 'b')
output, next_hidden = rnn(line_tensor[1], next_hidden)

print("For given characters 'A' and 'l'")
print(f"\nNext hidden state <{next_hidden.size()}> is:")
print(next_hidden)

print(f"\nLikelihood <{output.size()}> for each label is:")
print(output)
print("-"*20)

# Continue to iterate over "b", "e", "r", "t"
output, next_hidden = rnn(line_tensor[2], next_hidden)
output, next_hidden = rnn(line_tensor[3], next_hidden)
output, next_hidden = rnn(line_tensor[4], next_hidden)
output, next_hidden = rnn(line_tensor[5], next_hidden)
# At the end, the output is nLikelihood for each label for given character 'A', 'l', 'b', 'e', 'r', t'
# We will use this output for classification

print(f"\nFor given characters 'A', 'l', 'b', 'e', 'r', t'")
print(f"Likelihood <{output.size()}> for each label is:")
print(output)

As you can see the output is a ``<1 x n_categories>`` Tensor, where
every item is the likelihood of that category (higher is more likely).



# Workshop Tasks

## Task 1
Implement the above code using `for loop`

In [13]:
print("Running 'Albert' through the RNN using `for loop`")
line_tensor = lineToTensor('Albert')
hidden = torch.zeros(1, n_hidden)
# TODO


## Task 2

Write code using pytorch operators to convert these to probabilities by exponentiating them (ie, f(x) = exp(x)). Print the result. You should find that they are all around 5% - 6%.

In [0]:
# TODO

## Training

### Preparing for Training

Before going into training we should make a few helper functions. The
first is to interpret the output of the network, which we know to be a
likelihood of each category. We can use ``Tensor.topk`` to get the index
of the greatest value:




In [0]:
def categoryFromOutput(output):
    top_n, top_i = output.topk(1)
    category_i = top_i[0].item()
    return all_categories[category_i], category_i

print(categoryFromOutput(output))

We will also want a quick way to get a training example (a name and its
language). We use randomness here as training on the same instances in the same order can lead to worse results as we overfit that particular sequence of samples.




In [0]:
import random

def randomChoice(l):
    return l[random.randint(0, len(l) - 1)]

def randomTrainingExample():
    category = randomChoice(all_categories)
    line = randomChoice(category_lines[category])
    category_tensor = torch.tensor([all_categories.index(category)], dtype=torch.long)
    line_tensor = lineToTensor(line)
    return category, line, category_tensor, line_tensor

print("Here are 10 examples of randomly choosing data samples:")
for i in range(10):
    category, line, category_tensor, line_tensor = randomTrainingExample()
    print('category =', category, '/ line =', line)

### Training the Network

Now all it takes to train this network is show it a bunch of examples,
have it make guesses, and tell it if it's wrong.

For the loss function ``nn.NLLLoss`` is appropriate, since the last
layer of the RNN is ``nn.LogSoftmax``.




In [0]:
criterion = nn.NLLLoss()

Each loop of training will:

-  Create input and target tensors
-  Create a zeroed initial hidden state
-  Read each letter in and do the calculation, keeping the hidden state for the next letter
-  Compare final output to target
-  Back-propagate
-  Return the output and loss




In [0]:
learning_rate = 0.005 # If you set this too high, it might explode. If too low, it might not learn

def train(category_tensor, line_tensor):
    hidden = rnn.initHidden()

    rnn.zero_grad()

    for i in range(line_tensor.size()[0]):
        output, hidden = rnn(line_tensor[i], hidden)

    loss = criterion(output, category_tensor)
    loss.backward()

    # Add parameters' gradients to their values, multiplied by learning rate
    for p in rnn.parameters():
        p.data.add_(p.grad.data, alpha=-learning_rate)

    return output, loss.item()

Now we just have to run that with a bunch of examples. Since the
``train`` function returns both the output and loss we can print its
guesses and also keep track of loss for plotting. Since there are 1000s
of examples we print only every ``print_every`` examples, and take an
average of the loss.


In [0]:
import time
import math

n_iters = 50000
print_every = 2500
plot_every = 500

# Keep track of losses for plotting
current_loss = 0
all_losses = []

def timeSince(since):
    now = time.time()
    s = now - since
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

start = time.time()

for iter in range(1, n_iters + 1):
    category, line, category_tensor, line_tensor = randomTrainingExample()
    output, loss = train(category_tensor, line_tensor)
    current_loss += loss

    # Print ``iter`` number, loss, name and guess
    if iter % print_every == 0:
        guess, guess_i = categoryFromOutput(output)
        correct = '✓' if guess == category else '✗ (%s)' % category
        print('%d %d%% (%s) %.4f %s / %s %s' % (iter, iter / n_iters * 100, timeSince(start), loss, line, guess, correct))

    # Add current loss avg to list of losses
    if iter % plot_every == 0:
        all_losses.append(current_loss / plot_every)
        current_loss = 0

### Plotting the Results

Plotting the historical loss from ``all_losses`` shows the network
learning.

Note that learning is fast and fairly smooth at first, but then the improvements become smaller and more variable.




In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.figure()
plt.plot(all_losses)

## Evaluating the Results

Let's see how well the model is doing on the training data. We can get a reasonable estimate with just part of the data, so we'll run 1000 samples through the network with `evaluate()`, which is the same as `train()` minus the backpropagation.




In [0]:
# Just return an output given a line
def evaluate(line_tensor):
    hidden = rnn.initHidden()

    for i in range(line_tensor.size()[0]):
        output, hidden = rnn(line_tensor[i], hidden)

    return output

total = 1000
correct = 0
# Go through a bunch of examples and record which are correctly guessed
for i in range(total):
    category, line, category_tensor, line_tensor = randomTrainingExample()
    output = evaluate(line_tensor)
    guess, guess_i = categoryFromOutput(output)
    category_i = all_categories.index(category)
    if category_i == guess_i:
        correct += 1
print("{}%".format(100 * correct / total))

The score should be around 50-60%, which may seem low, but consider how tricky this task can be!




### Running on User Input

This function shows the output for a sample input you can provide.

In [0]:
def predict(input_line, n_predictions=3):
    print('\n> %s' % input_line)
    with torch.no_grad():
        output = evaluate(lineToTensor(input_line))

        # Get top N categories
        topv, topi = output.topk(n_predictions, 1, True)
        predictions = []

        for i in range(n_predictions):
            value = topv[0][i].item()
            category_index = topi[0][i].item()
            print('(%.2f) %s' % (value, all_categories[category_index]))
            predictions.append([value, all_categories[category_index]])

predict('Kummerfeld')
predict('Kay')

## Task 3

Above, we trained and tested on the same data. That is misleading, because the model saw those examples during training.

In this task:
1. Modify the data reading process to split the data randomly into a test set (\~10% of the data) and train set (\~90% of the data). Train and test again.
2. Modify `randomTrainingExample` to sample from your training data. Implement a `randomTestExample` to sample from your test data.
3. Create a new instance of the model.
4. Train that instance with the training data you created.
5. Test it with the test data you created.

In [0]:
# TODO

## Task 4

The model is entirely linear so far. Modify it to be the basic RNN introduced in lecture 3.

Note - you can do this task without doing Task 2. It is fine to report results on the training set (as the code below does). If you want to combine task 3 and 4 that's okay too.

In the process, also change the weight initialisation to set them to be random values uniformly distributed in the range (-sqrt(k), sqrt(k)) where k is 1/hidden_size.

The cells below contains all the key code from above for easier manipulation.

In [0]:
import string
all_letters = string.ascii_letters + " .,;’"
n_letters = len(all_letters)

# Read the category_lines dictionary, a list of names per language
category_lines = {}
all_categories = set()

with open("names.txt") as src:
    for line in src:
        parts = line.strip().split()
        category = parts[0]
        name = ' '.join(parts[1:])
        all_categories.add(category)
        category_lines.setdefault(category, []).append(name)
    
all_categories = sorted(list(all_categories))
n_categories = len(all_categories)

In [0]:
# Model and Inference

# Find letter index from all_letters, e.g. "a" = 0
def letterToIndex(letter):
    return all_letters.find(letter)

# Just for demonstration, turn a letter into a <1 x n_letters> Tensor
def letterToTensor(letter):
    tensor = torch.zeros(1, n_letters)
    tensor[0][letterToIndex(letter)] = 1
    return tensor

# Turn a line into a <line_length x 1 x n_letters>,
# or an array of one-hot letter vectors
def lineToTensor(line):
    tensor = torch.zeros(len(line), 1, n_letters)
    for li, letter in enumerate(line):
        tensor[li][0][letterToIndex(letter)] = 1
    return tensor

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RNN, self).__init__()

        self.hidden_size = hidden_size

        # Define the structure of the model
        self.i2h = nn.Linear(input_size + hidden_size, hidden_size)
        self.h2o = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)
        
        # Set the weights to some initial values
        self.init_weights()
        
    def init_weights(self):
        # Initialise the weights to be random values in the matrices and zero for the biases
        initrange = 0.5
        self.i2h.weight.data.uniform_(-initrange, initrange)
        self.i2h.bias.data.zero_()
        self.h2o.weight.data.uniform_(-initrange, initrange)
        self.h2o.bias.data.zero_()
        
    def initHidden(self):
        # Define the initial hidden state for an input as all zeros
        return torch.zeros(1, self.hidden_size)

    def forward(self, input_tensor, hidden):
        # Given an input, compute the steps defined by the model
        combined = torch.cat((input_tensor, hidden), 1)
        hidden = self.i2h(combined)
        output = self.h2o(hidden)
        output = self.softmax(output)
        return output, hidden

n_hidden = 128
rnn = RNN(n_letters, n_hidden, n_categories)

In [0]:
# Training

n_iters = 50000
print_every = 2500
plot_every = 500

criterion = nn.NLLLoss()

learning_rate = 0.005 # If you set this too high, it might explode. If too low, it might not learn

def categoryFromOutput(output):
    top_n, top_i = output.topk(1)
    category_i = top_i[0].item()
    return all_categories[category_i], category_i

def randomChoice(l):
    return l[random.randint(0, len(l) - 1)]

def randomTrainingExample():
    category = randomChoice(all_categories)
    line = randomChoice(category_lines[category])
    category_tensor = torch.tensor([all_categories.index(category)], dtype=torch.long)
    line_tensor = lineToTensor(line)
    return category, line, category_tensor, line_tensor

def train(category_tensor, line_tensor):
    hidden = rnn.initHidden()

    rnn.zero_grad()

    for i in range(line_tensor.size()[0]):
        output, hidden = rnn(line_tensor[i], hidden)

    loss = criterion(output, category_tensor)
    loss.backward()

    # Add parameters' gradients to their values, multiplied by learning rate
    for p in rnn.parameters():
        p.data.add_(p.grad.data, alpha=-learning_rate)

    return output, loss.item()

# Keep track of losses for plotting
current_loss = 0
all_losses = []

def timeSince(since):
    now = time.time()
    s = now - since
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

start = time.time()

for iter in range(1, n_iters + 1):
    category, line, category_tensor, line_tensor = randomTrainingExample()
    output, loss = train(category_tensor, line_tensor)
    current_loss += loss

    # Print ``iter`` number, loss, name and guess
    if iter % print_every == 0:
        guess, guess_i = categoryFromOutput(output)
        correct = '✓' if guess == category else '✗ (%s)' % category
        print('%d %d%% (%s) %.4f %s / %s %s' % (iter, iter / n_iters * 100, timeSince(start), loss, line, guess, correct))

    # Add current loss avg to list of losses
    if iter % plot_every == 0:
        all_losses.append(current_loss / plot_every)
        current_loss = 0

In [0]:
# Just return an output given a line
def evaluate(line_tensor):
    hidden = rnn.initHidden()

    for i in range(line_tensor.size()[0]):
        output, hidden = rnn(line_tensor[i], hidden)

    return output

total = 1000
correct = 0
# Go through a bunch of examples and record which are correctly guessed
for i in range(total):
    category, line, category_tensor, line_tensor = randomTrainingExample()
    output = evaluate(line_tensor)
    guess, guess_i = categoryFromOutput(output)
    category_i = all_categories.index(category)
    if category_i == guess_i:
        correct += 1
print("{}%".format(100 * correct / total))

## Resources

For more on RNNs, see:

- [Chapter 9 of J+M](https://web.stanford.edu/~jurafsky/slp3/9.pdf)
- [Chapter 7, section 6 of E](https://github.com/jacobeisenstein/gt-nlp-class/blob/master/notes/eisenstein-nlp-notes.pdf)
-  [The Unreasonable Effectiveness of Recurrent Neural
   Networks](https://karpathy.github.io/2015/05/21/rnn-effectiveness/) shows a bunch of real life examples
-  [Understanding LSTM
   Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) is about LSTMs specifically but also informative about RNNs in general